In [ ]:

import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define persistent directory for the project on Drive
PROJECT_DIR = '/content/drive/MyDrive/Progetto_Pneumonia_Shared'
os.makedirs(PROJECT_DIR, exist_ok=True)
MODEL_SAVE_PATH = os.path.join(PROJECT_DIR, 'best_pneumonia_resnet2.pth')

print(f"Persistent storage configured at: {PROJECT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent storage configured at: /content/drive/MyDrive/Progetto_Pneumonia_Shared


download dataset

In [ ]:
import zipfile

# WARNING: your Kaggle API credentials
os.environ['KAGGLE_USERNAME'] = "INSERT_YOUR_USERNAME"
os.environ['KAGGLE_KEY'] = "INSERT_YOUR_KEY"

dataset_name = "paultimothymooney/chest-xray-pneumonia"
data_dir = "./chest_xray_data"
zip_path = "chest-xray-pneumonia.zip"

print("--- Dataset Acquisition ---")
if not os.path.exists(data_dir):
    print("Downloading dataset via Kaggle API...")
    os.system(f"kaggle datasets download -d {dataset_name}")
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    os.remove(zip_path)
    print("Dataset downloaded and extracted successfully.")
else:
    print("Dataset already exists in the local Colab environment.")

--- Dataset Acquisition ---
Extracting files...
Dataset downloaded and extracted successfully.


Cella 3: Librerie e Configurazione Hardware

In [ ]:
import copy
import time
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import cv2
from PIL import Image

# Hardware configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")

Device in use: cuda


Cella 4: Data Pipeline & Augmentation Strategy

In [ ]:
class PneumoniaDataModule:
    """Gestisce il caricamento dei dati, lo split stratificato e le trasformazioni."""
    def __init__(self, data_dir="./chest_xray_data/chest_xray", batch_size=32, val_split=0.2):
        self.batch_size = batch_size
        self.val_split = val_split
        self.train_dir = os.path.join(data_dir, "train")
        self.test_dir = os.path.join(data_dir, "test")

    def get_transforms(self):
        train_transforms = transforms.Compose([
            transforms.Resize(240),
            transforms.CenterCrop(224),
            transforms.RandomRotation(15),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
        ])

        eval_transforms = transforms.Compose([
            transforms.Resize(240),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
        ])
        return train_transforms, eval_transforms

    def get_dataloaders(self):
        train_tf, eval_tf = self.get_transforms()
        base_dataset = datasets.ImageFolder(self.train_dir)
        targets = base_dataset.targets

        train_idx, val_idx = train_test_split(
            list(range(len(base_dataset))),
            test_size=self.val_split,
            stratify=targets,
            random_state=42
        )

        full_train_dataset = datasets.ImageFolder(self.train_dir, transform=train_tf)
        full_val_dataset = datasets.ImageFolder(self.train_dir, transform=eval_tf)

        dataloaders = {
            'train': DataLoader(Subset(full_train_dataset, train_idx), batch_size=self.batch_size, shuffle=True, num_workers=2),
            'val': DataLoader(Subset(full_val_dataset, val_idx), batch_size=self.batch_size, shuffle=False, num_workers=2),
            'test': DataLoader(datasets.ImageFolder(self.test_dir, transform=eval_tf), batch_size=self.batch_size, shuffle=False, num_workers=2)
        }
        return dataloaders

print("Data Module initialized.")

Data Module initialized.


Cella 5: Network Architecture

In [ ]:
class PneumoniaResNet(nn.Module):
    """Architettura ResNet-50 configurata rigorosamente per Grad-CAM."""
    def __init__(self):
        super(PneumoniaResNet, self).__init__()
        self.base_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)


        for name, param in self.base_model.named_parameters():
          ####"layer4" in name or add to the if below if you want to freeze the layer4 aswell
            if  "fc" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        num_ftrs = self.base_model.fc.in_features
        self.base_model.fc = nn.Linear(num_ftrs, 1)

    def forward(self, x):
        return self.base_model(x)

Cella 6: Motore XAI (Grad-CAM & MRS)

In [ ]:
class GradCAMEngine:
    """Genera heatmaps Grad-CAM e calcola l'MRS geometrico."""
    def __init__(self, model):
        self.model = model
        self.target_layer = self.model.base_model.layer4[-1]
        self.gradients = None
        self.activations = None

        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def get_heatmap(self, input_tensor):
        self.model.eval()
        input_tensor.requires_grad_()

        logits = self.model(input_tensor)
        self.model.zero_grad()
        logits.backward(retain_graph=True)

        gradients = self.gradients.detach()
        activations = self.activations.detach()

        weights = torch.mean(gradients, dim=[2, 3], keepdim=True)
        weighted_activations = activations * weights
        heatmap = torch.sum(weighted_activations, dim=1).squeeze()
        heatmap = F.relu(heatmap)

        heatmap_max = torch.max(heatmap)
        if heatmap_max > 0:
            heatmap /= heatmap_max

        return heatmap.cpu().numpy()

    def calculate_mrs(self, heatmap):
        """Calcola l'MRS tagliano via i bordi anatomici spurii."""
        h, w = heatmap.shape
        margin_h = int(h * 0.08)  # 8%
        margin_w = int(w * 0.05)  # 5%
        relevance_inside = np.sum(heatmap[margin_h:h-margin_h, margin_w:w-margin_w])
        total_relevance = np.sum(heatmap)
        if total_relevance == 0:
            return 0.0
        return relevance_inside / total_relevance

Cella 7: Training Loop (Minimizzazione della Loss)

In [ ]:
class StandardModelTrainer:
    """Motore di addestramento basato rigorosamente sulla Loss. Logga l'MRS ma non lo usa per l'ottimizzazione."""
    def __init__(self, model, dataloaders, device):
        self.model = model.to(device)
        self.dataloaders = dataloaders
        self.device = device
        self.criterion = nn.BCEWithLogitsLoss()
        self.grad_cam = GradCAMEngine(self.model)

    def train(self, num_epochs=50, base_lr=1e-5, fc_lr=1e-3, weight_decay=1e-4):
        # Differiantial Learning Rate
        optimizer = torch.optim.Adam([
            {'params': self.model.base_model.layer4.parameters(), 'lr': base_lr},
            {'params': self.model.base_model.fc.parameters(), 'lr': fc_lr}
        ], weight_decay=weight_decay)

        best_model_wts = copy.deepcopy(self.model.state_dict())
        best_val_loss = float('inf')

        ### history structure
        history = {'train_acc': [], 'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_mrs': []}
        start_time = time.time()

        for epoch in range(num_epochs):
            print(f'Epoch {epoch+1}/{num_epochs} ', end="")

            # --- TRAINING PHASE ---
            self.model.train()
            running_loss, running_corrects_train = 0.0, 0

            for inputs, labels in self.dataloaders['train']:
                inputs, labels = inputs.to(self.device), labels.float().unsqueeze(1).to(self.device)
                optimizer.zero_grad()
                logits = self.model(inputs)
                loss = self.criterion(logits, labels)
                preds = (torch.sigmoid(logits) > 0.5).float()

                loss.backward()
                optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects_train += torch.sum(preds == labels.data).item()

            epoch_train_loss = running_loss / len(self.dataloaders['train'].dataset)
            epoch_train_acc = running_corrects_train / len(self.dataloaders['train'].dataset)

            # --- VALIDATION PHASE ---
            self.model.eval()
            running_val_loss, running_corrects_val = 0.0, 0
            epoch_mrs_sum = 0.0

            for inputs, labels in self.dataloaders['val']:
                inputs, labels = inputs.to(self.device), labels.float().unsqueeze(1).to(self.device)

                with torch.no_grad():
                    logits = self.model(inputs)
                    loss = self.criterion(logits, labels)
                    preds = (torch.sigmoid(logits) > 0.5).float()

                    running_val_loss += loss.item() * inputs.size(0)
                    running_corrects_val += torch.sum(preds == labels.data).item()

                # Calcolo MRS ai soli fini statistici
                with torch.set_grad_enabled(True):
                    sample_input = inputs[0:1].clone().detach().requires_grad_(True)
                    heatmap = self.grad_cam.get_heatmap(sample_input)
                    epoch_mrs_sum += self.grad_cam.calculate_mrs(heatmap)

            epoch_val_loss = running_val_loss / len(self.dataloaders['val'].dataset)
            epoch_val_acc = running_corrects_val / len(self.dataloaders['val'].dataset)
            epoch_mrs = epoch_mrs_sum / len(self.dataloaders['val'])

            history['train_acc'].append(epoch_train_acc)
            history['train_loss'].append(epoch_train_loss)
            history['val_loss'].append(epoch_val_loss)
            history['val_acc'].append(epoch_val_acc)
            history['val_mrs'].append(epoch_mrs)

            print(f'- T.Loss: {epoch_train_loss:.4f} | V.Loss: {epoch_val_loss:.4f} | V.Acc: {epoch_val_acc:.4f} | MRS: {epoch_mrs:.4f}')

            # Salvataggio oggettivo basato sulla Validation Loss
            if epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                best_model_wts = copy.deepcopy(self.model.state_dict())

        print(f'\nTraining complete in {(time.time() - start_time) // 60:.0f}m')
        self.model.load_state_dict(best_model_wts)
        return self.model, history

Cella 8: Esecuzione e Plotting

In [ ]:
data_module = PneumoniaDataModule()
dataloaders = data_module.get_dataloaders()

print("Initializing ResNet-50 structure...")
model = PneumoniaResNet().to(device)

if os.path.exists(MODEL_SAVE_PATH):
    print("\n PRE-TRAINED MODEL FOUND ON GOOGLE DRIVE!")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    trained_model = model
    print("Model ready for clinical inference. (Skipping training plots)")
else:
    print("\n No saved model found. Initiating training process...")
    trainer = StandardModelTrainer(model, dataloaders, device)

    trained_model, history = trainer.train(
        num_epochs=50,
        base_lr=1e-5,
        fc_lr=1e-3,
        weight_decay=1e-4
    )
    torch.save(trained_model.state_dict(), MODEL_SAVE_PATH)
    print(f"\n💾 Training complete! Model weights saved to: {MODEL_SAVE_PATH}")



Initializing ResNet-50 structure...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]



✅ PRE-TRAINED MODEL FOUND ON GOOGLE DRIVE!
Model ready for clinical inference. (Skipping training plots)


In [ ]:
    epochs = range(1, len(history['val_acc']) + 1)
    plt.figure(figsize=(10, 8))

    # Grafico 1: Accuratezza
    plt.subplot(2, 2, 1)
    plt.plot(epochs, history['train_acc'], color='#314cb6', label='Train Accuracy')
    plt.plot(epochs, history['val_acc'], color='#43a047', label='Validation Accuracy')
    plt.title('Model Accuracy over 50 Epochs with Grad-CAM')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True, linestyle='-', alpha=0.5)
    plt.legend()

    # Grafico 2: Loss
    plt.subplot(2, 2, 2)
    plt.plot(epochs, history['train_loss'], color='#d32f2f', label='Train Loss')
    plt.plot(epochs, history['val_loss'], color='#fbc02d', label='Validation Loss')
    plt.title('Model Loss over 50 Epochs with Grad-CAM')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True, linestyle='-', alpha=0.5)
    plt.legend()

    # Grafico 3: Scatter Plot Accuracy vs MRS
    plt.subplot(2, 1, 2)
    plt.scatter(history['val_acc'], history['val_mrs'], color='#38969f', s=35, alpha=0.8, label='Accuracy vs Interpretability (MRS)')
    plt.title('Accuracy vs Interpretability (MRS) with Grad-CAM')
    plt.xlabel('Validation Accuracy')
    plt.ylabel('Interpretability (MRS)')
    plt.grid(True, linestyle='-', alpha=0.5)
    plt.legend(loc='upper left')

    plt.tight_layout()
    plt.show()

NameError: name 'history' is not defined

Cella 9: Calcolo della Soglia Ottimale (ROC su Validation Set)

In [ ]:
print("Calcolo della Soglia Ottimale sul Validation Set")

trained_model.eval()
val_probs = []
val_trues = []

with torch.no_grad():
    for inputs, labels in dataloaders['val']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = trained_model(inputs)
        probs = torch.sigmoid(outputs).cpu().numpy()

        val_probs.extend(probs)
        val_trues.extend(labels.cpu().numpy())

fpr, tpr, thresholds = roc_curve(val_trues, val_probs)
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
optimal_threshold = thresholds[best_idx]

print(f"Soglia Ottimale trovata: {optimal_threshold:.4f}")

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label='ROC Curve')
plt.scatter(fpr[best_idx], tpr[best_idx], color='red', marker='o', s=100, label=f'Optimal Threshold ({optimal_threshold:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve - Validation Set')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

cekka10: valutazione clinica

In [ ]:
def evaluate_clinical_metrics(model, test_loader, device, threshold):
    model.eval()
    all_labels, all_probs, all_preds = [], [], []

    print("Evaluating optimal model on the Test Set...")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels_np = labels.cpu().numpy()
            logits = model(inputs)
            probs = torch.sigmoid(logits).cpu().numpy().squeeze()
            if probs.ndim == 0: probs = np.expand_dims(probs, axis=0)
            preds = (probs > threshold).astype(float)

            all_labels.extend(labels_np)
            all_probs.extend(probs)
            all_preds.extend(preds)

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    all_preds = np.array(all_preds)

    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal','Pneumonia'], yticklabels=['Normal','Pneumonia'])
    plt.xlabel('Predicted')
    plt.ylabel('Target')
    plt.title('CONFUSION MATRIX')
    plt.show()

    accuracy = accuracy_score(all_labels, all_preds)
    auc_roc = roc_auc_score(all_labels, all_probs)
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print("\n" + "="*45)
    print("📊 CLINICAL PERFORMANCE METRICS (TEST SET)")
    print("="*45)
    print(f"Accuracy:    {accuracy:.4f}")
    print(f"Sensitivity: {sensitivity:.4f} (True Positive Rate)")
    print(f"Specificity: {specificity:.4f} (True Negative Rate)")
    print(f"AUC-ROC:     {auc_roc:.4f}")
    print("="*45)

evaluate_clinical_metrics(trained_model, dataloaders['test'], device, optimal_threshold)

cella11: Valutazione MRS globle sul test set

In [ ]:
print("Valutazione Quantitativa dell'Interpretabilità (Grad-CAM su Test Set)")

xai_engine = GradCAMEngine(trained_model)
trained_model.eval()

total_mrs = 0.0
valid_samples = 0

for inputs, labels in dataloaders['test']:
    inputs = inputs.to(device)


    for i in range(inputs.size(0)):
        single_input = inputs[i:i+1].clone().detach().requires_grad_(True)
        heatmap = xai_engine.get_heatmap(single_input)
        mrs_score = xai_engine.calculate_mrs(heatmap)
        total_mrs += mrs_score
        valid_samples += 1

final_average_mrs = total_mrs / valid_samples

print("="*45)
print(f"✅ FINAL INTERPRETABILITY MRS (GRAD-CAM): {final_average_mrs:.4f}")
print("="*45)

Cella 12: Visualizzazione (Original vs Grad-CAM)

In [ ]:
print("Extracting 20 random images for Original vs Grad-CAM comparison")

xai_engine = GradCAMEngine(trained_model)
test_dataset = dataloaders['test'].dataset
random_indices = random.sample(range(len(test_dataset)), 20)

fig, axes = plt.subplots(5, 8, figsize=(28, 20))
axes = axes.flatten()

for i, idx in enumerate(random_indices):
    img_tensor, true_label = test_dataset[idx]

    input_tensor = img_tensor.unsqueeze(0).to(device)
    input_tensor.requires_grad_()
    heatmap = xai_engine.get_heatmap(input_tensor)

    trained_model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(trained_model(input_tensor)).item()

    pred_label = 1 if prob > optimal_threshold else 0
    true_str = "PNEU" if true_label == 1 else "NORM"
    pred_str = "PNEU" if pred_label == 1 else "NORM"
    color = 'green' if true_label == pred_label else 'red'

    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()

    heatmap_resized = cv2.resize(heatmap, (img_np.shape[1], img_np.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = np.float32(heatmap_color) / 255.0
    heatmap_color = heatmap_color[:, :, ::-1]

    superimposed_img = heatmap_color * 0.4 + img_np * 0.6
    superimposed_img = np.clip(superimposed_img, 0, 1)

    ax_orig = axes[i * 2]
    ax_orig.imshow(img_np)
    ax_orig.set_title(f"ORIGINAL\nTrue: {true_str}", color='black', fontsize=11, fontweight='bold')
    ax_orig.axis('off')

    ax_cam = axes[i * 2 + 1]
    ax_cam.imshow(superimposed_img)
    ax_cam.set_title(f"GRAD-CAM\nPred: {pred_str} ({prob:.2f})", color=color, fontsize=11, fontweight='bold')
    ax_cam.axis('off')

plt.tight_layout()
plt.subplots_adjust(wspace=0.1, hspace=0.3)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

dataset = dataloaders['val'].dataset
idx = random.randint(0, len(dataset) - 1)
img_tensor, label = dataset[idx]

img_np = img_tensor.permute(1, 2, 0).numpy()
h, w, _ = img_np.shape

margin_h = int(h * 0.08)
margin_w = int(w * 0.05)

box_width = w - (2 * margin_w)
box_height = h - (2 * margin_h)

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.imshow(img_np)

rect_mrs = patches.Rectangle(
    (margin_w, margin_h),
    box_width,
    box_height,
    linewidth=3,
    edgecolor='red',
    facecolor='none',

)
ax.add_patch(rect_mrs)

ax.add_patch(patches.Rectangle((0, 0), w, margin_h, color='black', alpha=0.4))
ax.add_patch(patches.Rectangle((0, h-margin_h), w, margin_h, color='black', alpha=0.4))
ax.add_patch(patches.Rectangle((0, margin_h), margin_w, box_height, color='black', alpha=0.4))
ax.add_patch(patches.Rectangle((w-margin_w, margin_h), margin_w, box_height, color='black', alpha=0.4))

class_name = "PNEUMONIA" if label == 1 else "NORMAL"
ax.set_title('Example of BoundingBox used')
ax.axis('off')
ax.legend(loc='lower left')

plt.tight_layout()
plt.show()